# Combined Audio Deepfake Detection (ASVspoof + FakeAVCeleb)


In [ ]:
!pip install librosa numpy pandas scikit-learn tensorflow tqdm

In [ ]:
import os, json
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import tensorflow as tf
from tensorflow.keras import layers, models


In [ ]:
def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=16000, duration=2)
        spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
        log_spec = librosa.power_to_db(spec)
        return log_spec
    except:
        return None


In [ ]:
# ASVspoof loading
asv_path = '/content/ASVspoof2019_LA_train/flac/'
label_file = '/content/ASVspoof2019.LA.cm.train.trn.txt'
cols = ['speaker','file','env','attack','label']
df = pd.read_csv(label_file, sep=' ', names=cols)

X_asv, y_asv = [], []
for _, row in tqdm(df.iterrows(), total=2000):
    file_path = os.path.join(asv_path, row['file'] + '.flac')
    if os.path.exists(file_path):
        feat = extract_features(file_path)
        if feat is not None:
            X_asv.append(feat)
            y_asv.append(1 if row['label']=='spoof' else 0)


In [ ]:
# FakeAVCeleb loading
fakeav_path = '/content/FakeAVCeleb/videos/'
meta_path = '/content/FakeAVCeleb/metadata.json'

with open(meta_path) as f:
    metadata = json.load(f)

X_fake, y_fake = [], []
for item in tqdm(metadata[:2000]):
    file_path = os.path.join(fakeav_path, item['file'])
    if os.path.exists(file_path):
        feat = extract_features(file_path)
        if feat is not None:
            X_fake.append(feat)
            y_fake.append(1 if item['audio_label']=='fake' else 0)


In [ ]:
X = np.array(X_asv + X_fake)
y = np.array(y_asv + y_fake)
X = X[..., np.newaxis]
X, y = shuffle(X, y, random_state=42)
print('Total samples:', len(X))


In [ ]:
model = models.Sequential([
    layers.Conv2D(32,(3,3),activation='relu',input_shape=X.shape[1:]),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64,(3,3),activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(128,(3,3),activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1,activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))


In [ ]:
model.save_weights('audio.weights.h5')
